In [19]:
import pandas as pd
import duckdb
from pathlib import Path

In [20]:
project_root: Path = Path.cwd().parent
silver_csv_path: Path = project_root / "data" / "silver" / "clean_cafe_sales.csv"
gold_dir = project_root / "data" / "gold"

df_silver: pd.DataFrame = pd.read_csv(silver_csv_path)

**Dimensión de Productos (dim_item)**

In [21]:
query_dim_item = """
    SELECT 
        ROW_NUMBER() OVER() As item_id,
        item as item_name
    FROM (SELECT DISTINCT item FROM df_silver WHERE item is NOT NULL)
"""

dim_item: pd.DataFrame = duckdb.query(query_dim_item).df()
dim_item

,item_id,item_name
0,1,Cookie
1,2,Juice
2,3,Coffee
3,4,Tea
4,5,Smoothie
5,6,Sandwich
6,7,Cake
7,8,Salad


**Dimensión de Pagos (dim_payment)**

In [22]:
query_dim_payment = """
    SELECT 
        ROW_NUMBER() OVER() AS payment_id,
        payment_method
    FROM (SELECT DISTINCT payment_method FROM df_silver WHERE payment_method IS NOT NULL)
"""

dim_payment : pd.DataFrame = duckdb.query(query_dim_payment).df()
dim_payment

,payment_id,payment_method
0,1,Cash
1,2,Digital Wallet
2,3,Unknown
3,4,Credit Card


**Dimensión de tipo de consumo (dim_location)**

In [23]:
query_dim_location = """
    SELECT 
        ROW_NUMBER() OVER() AS location_id,
        location AS location_name
    FROM (SELECT DISTINCT location FROM df_silver WHERE location IS NOT NULL)
"""

dim_location : pd.DataFrame = duckdb.query(query_dim_location).df()
dim_location

,location_id,location_name
0,1,In-Store
1,2,Unknown
2,3,Takeaway


**Dimensión de Fechas (dim_date)**

In [24]:
query_dim_date = """
    SELECT DISTINCT
        CAST( STRFTIME( CAST(transaction_date AS DATE), '%Y%m%d' ) AS INTEGER ) AS date_id,
        CAST(transaction_date AS DATE) AS full_date,
        day_of_week,
        month_name,
        is_weekend
    FROM df_silver
    ORDER BY date_id
"""

dim_date : pd.DataFrame = duckdb.query(query_dim_date).df()
dim_date

,date_id,full_date,day_of_week,month_name,is_weekend
0,20230101,2023-01-01,Sunday,January,True
1,20230102,2023-01-02,Monday,January,False
2,20230103,2023-01-03,Tuesday,January,False
3,20230104,2023-01-04,Wednesday,January,False
4,20230105,2023-01-05,Thursday,January,False
...,...,...,...,...,...
360,20231227,2023-12-27,Wednesday,December,False
361,20231228,2023-12-28,Thursday,December,False
362,20231229,2023-12-29,Friday,December,False
363,20231230,2023-12-30,Saturday,December,True


### Creación de la Tabla de Hechos (fact_sales)

In [25]:
query_fact_sales = """
    SELECT
        -- business key (id original para trazabilidad)
        s.transaction_id,
        
        -- foreign Keys (conectan con las dimensiones)
        CAST(STRFTIME(CAST(s.transaction_date AS DATE), '%Y%m%d') AS INTEGER) AS date_id,
        i.item_id,
        l.location_id,
        p.payment_id,

        -- measures (métricas del negocio)
        s.quantity,
        s.price_per_unit,
        s.total_spent
    
    FROM df_silver AS s
    -- cruzamos con la dimensión de productos
    LEFT JOIN dim_item AS i 
        ON s.item = i.item_name
    -- cruzamos con la dimensión de modalidad de pedido
    LEFT JOIN dim_location AS l 
        ON s.location = l.location_name
    -- cruzamos con la dimensión de pagos
    LEFT JOIN dim_payment AS p 
        ON s.payment_method = p.payment_method
"""

fact_sales: pd.DataFrame = duckdb.query(query_fact_sales).df()

fact_sales.head()

,transaction_id,date_id,item_id,location_id,payment_id,quantity,price_per_unit,total_spent
0,TXN_1961373,20230908,3,3,4,2,2.0,4.0
1,TXN_4977031,20230516,7,1,1,4,3.0,12.0
2,TXN_4271903,20230719,1,1,4,4,1.0,4.0
3,TXN_7034554,20230427,8,2,3,2,5.0,10.0
4,TXN_3160411,20230611,3,1,2,2,2.0,4.0


In [26]:
print(f"Total de filas: {len(fact_sales)}")
print(f"df_silver == fact_sales: {len(df_silver) == len(fact_sales)}")

Total de filas: 9067
df_silver == fact_sales: True


### Exportamos las Dimensiones (Catálogos) a formato Parquet

In [27]:
gold_dir

PosixPath('/home/gaby/development-workspace/cafesales_etl_project/data/gold')

In [30]:
# Exportamos las Dimensiones usando SQL nativo de DuckDB
# DuckDB detecta automáticamente tus DataFrames de Pandas y los exporta a Parquet
duckdb.query(f"COPY dim_item TO '{str(gold_dir / 'dim_item.parquet')}' (FORMAT PARQUET)")
duckdb.query(f"COPY dim_location TO '{str(gold_dir / 'dim_location.parquet')}' (FORMAT PARQUET)")
duckdb.query(f"COPY dim_payment TO '{str(gold_dir / 'dim_payment.parquet')}' (FORMAT PARQUET)")
duckdb.query(f"COPY dim_date TO '{str(gold_dir / 'dim_date.parquet')}' (FORMAT PARQUET)")
duckdb.query(f"COPY fact_sales TO '{str(gold_dir / 'fact_sales.parquet')}' (FORMAT PARQUET)")
